[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%204/retrieval-hackathon/search_hackathon.ipynb)

# The Retrieval Hackathon — Meaning vs. Words

**Week 4, Lecture 7.**

**The challenge.** Build the best search engine for this corpus of ~800 passages. Beat the keyword baseline, then beat each other. Same corpus, same queries, same metric for every team — *how* you build the retriever is wide open.

**The metric.** For each query, your retriever ranks the passages; we check where the correct one lands. **MRR@10** (higher = the right passage sits near the top) is the headline score, alongside **Recall@5** and your **index build time** (the cost axis).

**Two rounds — and the goalposts move.**
- **Round 1 (the sprint):** the leaderboard is ranked by your **Main MRR** — the literal queries, where keyword search is strong.
- **Round 2 (the reveal):** the ranking switches to your **Combined MRR** = (Main + Curveball) / 2 — now the paraphrased queries count too, and that's where semantic search earns its keep.

Every `submit()` shows both scores from the start, so you always know where you stand — but only **Round 1 counts until your instructor calls Round 2.**

## About the data

The corpus is **800 short passages** — paragraphs from Wikipedia articles across many topics (history, science, sports, geography, and more), drawn from the SQuAD reading-comprehension dataset. Each query is a natural-language **question**, and exactly one passage is its correct answer.

Three query sets — all bundled in the course repo, so there's nothing to download:

- **`main`** — 150 *literal* questions (their wording overlaps the answer passage).
- **`curveball`** — 30 *paraphrased* questions (same meaning, different words — the semantic test).
- **`dev`** — 100 questions for *tuning* your settings (don't score the leaderboard on these).

Your job: rank the 800 passages so the correct one lands at the top.

## Rules

- **Teams.** Your hackathon team is your **Final Team Project group** (Team 1, 2, 4, 5, 7, or 8). One shared notebook per team.
- **Any model, any method — best results win.** No assigned approaches. **Round 1** ranks on **Main MRR**; **Round 2** ranks on **Combined MRR** = (Main + Curveball) / 2.
- **Rank from the query only.** Your retriever must be a function `search(query) -> ranked passage ids` that scores passages from the *query text and the corpus*. You may **not** read `gold_id`, and you may **not** build a query→answer lookup. *Test: if it wouldn't work on a brand-new question typed live, it's not allowed* — this is the retrieval version of "don't touch the test set."
- **Tune on `dev`, not the leaderboard.** Choose your settings using the `dev` queries; score `main`/`curveball` only when you `submit()`. Tuning by maximizing the leaderboard queries is the train-on-test trap in disguise.

> **Spirit of the game:** build a real retriever, not a lookup table. Beat the baseline honestly, then explain *why* you won.

## Section 0 — Setup: load the data and the scoring (everyone runs this first)

In [ ]:
import json, time, urllib.request
import numpy as np

BASE = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%204/retrieval-hackathon"
def load_jsonl(name):
    try:
        raw = urllib.request.urlopen(f"{BASE}/{name}").read().decode()   # from the course repo
    except Exception:
        raw = open(name).read()                                          # local fallback
    return [json.loads(l) for l in raw.splitlines() if l.strip()]

corpus  = [r["text"] for r in sorted(load_jsonl("corpus.jsonl"), key=lambda r: r["id"])]
main    = load_jsonl("queries.jsonl")       # literal questions
curve   = load_jsonl("curveball.jsonl")     # paraphrased questions (the plot twist)
dev     = load_jsonl("dev.jsonl")           # TUNING set -- optimize on this, never on main/curve
print("corpus:", len(corpus), " main:", len(main), " curveball:", len(curve), " dev:", len(dev))

# --- scoring: your retriever is a function  query(str) -> list of doc ids, best first ---
def score(retriever, queries):
    r1 = r5 = mrr = 0
    for q in queries:
        ranked = retriever(q["query"])
        rank = next((j+1 for j,idx in enumerate(ranked) if idx == q["gold_id"]), 999)
        r1  += rank == 1
        r5  += rank <= 5
        mrr += (1.0/rank) if rank <= 10 else 0
    n = len(queries)
    return {"r@1": r1/n, "r@5": r5/n, "mrr": mrr/n}

def evaluate(retriever):
    return score(retriever, main), score(retriever, curve)

def submit(team, method, retriever, build_seconds):
    m, c = evaluate(retriever)
    combined = (m["mrr"] + c["mrr"]) / 2
    print("\n" + "="*58)
    print("  LEADERBOARD ROW")
    print("="*58)
    print(f"  Team               : {team}")
    print(f"  Method             : {method}")
    print(f"  ROUND 1  Main MRR  : {m['mrr']:.3f}   <- the sprint ranks on this")
    print(f"  ROUND 2  Combined  : {combined:.3f}   <- the reveal ranks on this")
    print(f"           = (main {m['mrr']:.3f} + curveball {c['mrr']:.3f}) / 2")
    print(f"  Recall@5           : {m['r@5']:.2f}")
    print(f"  Index build (s)    : {build_seconds:.1f}")
    print("-"*58)
    print("  Paste this tab-separated line into the leaderboard:")
    print(f"  {team}\t{method}\t{m['mrr']:.3f}\t{c['mrr']:.3f}\t{m['r@5']:.2f}\t{build_seconds:.1f}")

## Your baseline to beat: TF-IDF keyword search (given)

In [ ]:
# Baseline: TF-IDF keyword search. This is the interface your retriever must match:
#   a function that takes a query string and returns doc ids ranked best-first.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

t0 = time.time()
tfidf = TfidfVectorizer(stop_words="english")
doc_vectors = tfidf.fit_transform(corpus)          # <- this is the 'index'
build_seconds = time.time() - t0

def tfidf_search(query):
    sims = cosine_similarity(tfidf.transform([query]), doc_vectors)[0]
    return list(np.argsort(-sims))

m, c = evaluate(tfidf_search)
print(f"TF-IDF baseline  ->  main MRR {m['mrr']:.3f}   |   curveball MRR {c['mrr']:.3f}")
# When you're ready, post it:  submit('Team ?', 'TF-IDF baseline', tfidf_search, build_seconds)

## Build your retriever

A **retriever** is a function: give it a question (a string), it returns the passage numbers ranked best-first. You already built one — `tfidf_search`, the baseline. Every retriever follows the same shape:

```python
def my_search(query):        # query is a question, like "when did the war end?"
    ...                      # score/rank the 800 passages
    return ranked_ids        # a list of passage numbers, best match first
```

To check and post any retriever:

```python
m, c = evaluate(my_search)                                # your two scores
submit("Team 1", "my method", my_search, build_seconds)  # prints your leaderboard row
```

**New to Python? Do these in order — each cell below is complete. Copy it, run it, submit it.**

- **Step 1** — submit the baseline you already have (a valid entry to get on the board).
- **Step 2** — upgrade to semantic search. Watch the curveball score jump.
- **Step 3** — go to the next section and tune the hybrid dial (usually the winner).

Best combined score wins, so keep improving.

In [ ]:
# STEP 1 -- submit the baseline you already built above.
# (tfidf_search and build_seconds both come from the baseline cell -- run that first.)
m, c = evaluate(tfidf_search)
print(f"baseline  ->  main MRR {m['mrr']:.3f} | curveball MRR {c['mrr']:.3f}")

submit("Team 1", "TF-IDF baseline", tfidf_search, build_seconds)   # <-- change "Team 1" to your team

In [ ]:
# STEP 2 -- semantic search. It matches meaning, not just matching words.
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np, time

t0 = time.time()
# --- Pick your embedding model: uncomment ONE line (numbers validated on this data) ---
MODEL = "all-MiniLM-L6-v2"            # fastest + smallest, build ~5s   <-- start here
# MODEL = "multi-qa-MiniLM-L6-cos-v1"  # tuned for search, build ~12s
# MODEL = "all-mpnet-base-v2"          # most accurate, ~6x slower + bigger, build ~30s
model = SentenceTransformer(MODEL)
doc_embeddings = model.encode(corpus, normalize_embeddings=True)  # encode all 800 passages = your index
build_seconds = time.time() - t0

def my_search(query):
    q = model.encode([query], normalize_embeddings=True)[0]       # turn the question into a vector
    scores = doc_embeddings @ q                                   # similarity to every passage
    return list(np.argsort(-scores))                              # best match first

m, c = evaluate(my_search)
print(f"semantic  ->  main MRR {m['mrr']:.3f} | curveball MRR {c['mrr']:.3f}")

submit("Team 1", "semantic (MiniLM)", my_search, build_seconds)   # <-- change "Team 1" to your team

Notice the **curveball score jumps** — paraphrases need meaning, not matching words — even though the main score may dip a little. The next section shows how to **blend keyword + semantic** and tune the mix; that hybrid is usually what wins.

**Want to experiment?** Not every idea helps. Try adding word-pairs (`ngram_range=(1, 2)`) or BM25 to a keyword search and check on `dev` whether it actually beats the plain baseline — finding what *doesn't* help is part of the game.

### Toolbox — swap in other models & methods

**Change your embedding model** — in Step 2, just uncomment a different `MODEL =` line:

| Model | Main | Curve | Notes |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 0.74 | 0.66 | fast, small (default) |
| `multi-qa-MiniLM-L6-cos-v1` | 0.73 | 0.68 | tuned for search |
| `all-mpnet-base-v2` | 0.78 | 0.68 | most accurate, ~6x slower/bigger |
| `intfloat/e5-small-v2` | **0.85** | **0.82** | strongest here, small & fast — see snippet below |

**E5-small — the strongest single model on this data.** It needs `"query: "` / `"passage: "` prefixes (it was trained to know which side is the question — plain encoding underperforms). Copy into a cell:

```python
from sentence_transformers import SentenceTransformer
import numpy as np, time
t0 = time.time()
e5 = SentenceTransformer("intfloat/e5-small-v2")
doc_emb = e5.encode(["passage: " + d for d in corpus], normalize_embeddings=True)  # index
build_seconds = time.time() - t0
def my_search(query):
    q = e5.encode(["query: " + query], normalize_embeddings=True)[0]
    return list(np.argsort(-(doc_emb @ q)))
m, c = evaluate(my_search)
print(f"E5-small  ->  main MRR {m['mrr']:.3f} | curveball MRR {c['mrr']:.3f}")
# submit("Team 1", "E5-small", my_search, build_seconds)
```

**BM25** — a classic keyword ranker. Copy into a new cell:

```python
!pip -q install rank_bm25
from rank_bm25 import BM25Okapi
import numpy as np
bm25 = BM25Okapi([doc.lower().split() for doc in corpus])
def my_search(query):
    return list(np.argsort(-bm25.get_scores(query.lower().split())))
```

**Cross-encoder re-ranker** — re-scores the top results with a heavier model. Slower per query, but a big accuracy jump. Copy into a cell *after* you already have a `my_search`:

```python
from sentence_transformers import CrossEncoder
import numpy as np
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
_base = my_search                       # your current retriever
def reranked_search(query, topn=50):
    ids = _base(query)[:topn]
    order = np.argsort(-reranker.predict([(query, corpus[i]) for i in ids]))
    return [ids[j] for j in order] + _base(query)[topn:]
# submit("Team 1", "rerank", reranked_search, build_seconds)
```

Every method uses the same `search(query) -> ranked ids` shape, so you can `submit()` any of them.

## Tune it — find the dial (optional, but this is where the leaderboard is won)

Most retrievers have knobs. The most powerful is the **hybrid dial** `alpha`: `0.0` = pure keyword, `1.0` = pure semantic. The best setting is usually *in between* — and you have to find it.

**The one tuning rule (read this):** tune on the **`dev`** set. Do **not** pick your knobs by maximizing the score on `main`/`curveball` — that's tuning on the test set, the same train-on-test trap from the bake-off, and it won't generalize. Find your best settings on `dev`, then score the leaderboard once with `submit()`.

In [ ]:
# A tunable hybrid retriever. Build the two indexes once, then dial `alpha`.
!pip -q install sentence-transformers 2>/dev/null
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np, time

t0 = time.time()
tf = TfidfVectorizer(stop_words="english"); Xtf = tf.fit_transform(corpus)
enc = SentenceTransformer("all-MiniLM-L6-v2")
Edocs = enc.encode(corpus, normalize_embeddings=True)     # <- the semantic index
build_seconds = time.time() - t0

def _nz(v):
    v = v - v.min(); mx = v.max(); return v/mx if mx > 0 else v

def make_hybrid(alpha):
    def search(query):
        kw = _nz(cosine_similarity(tf.transform([query]), Xtf)[0])
        se = _nz(Edocs @ enc.encode([query], normalize_embeddings=True)[0])
        return list(np.argsort(-(alpha*se + (1-alpha)*kw)))
    return search

# Sweep the dial on DEV only, then pick the winner:
for a in [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]:
    print(f"alpha={a:.1f}   dev MRR={score(make_hybrid(a), dev)['mrr']:.3f}")

# best_alpha = 0.8                      # <- set to your best dev alpha
# submit("Team X", f"hybrid a={best_alpha}", make_hybrid(best_alpha), build_seconds)

**Go further (all tuned on `dev`):** swap in a stronger encoder (`all-mpnet-base-v2`), tune BM25's `k1`/`b`, adjust TF-IDF `ngram_range`/`min_df`, or take the top 50 and re-rank with a cross-encoder. Every knob is fair game — **best combined score wins.**

## Post to the leaderboard, then we read it together

Paste your tab-separated row into the shared class leaderboard:

**[Retrieval Hackathon Leaderboard](https://docs.google.com/spreadsheets/d/18asBX7QYPLyA5x2jGoq2tswkoY-dPTxjWdKHBmBkNw8/edit)**

**Bring back an answer to:**
1. Where did keyword search win, and where did it fall apart?
2. How much did the curveball (paraphrase) round change the ranking versus the main round?
3. Where does your retriever sit on the quality-vs-cost frontier?

*ISBA 2411 - Natural Language Processing & AI - Summer 2026*